# 01 · Prepare the GazeCapture dataset (M2)

Runs MediaPipe Face Landmarker over every phone/portrait GazeCapture frame and builds the training manifest.

* **Resumable:** results are written per subject to Google Drive. If the Colab session dies, re-run all cells; finished subjects are skipped.
* **Fast I/O:** the archive is extracted to the VM's local disk (reading frames from Drive is very slow).
* Expected output on Drive: `eyegaze/landmarks/*.parquet`, `eyegaze/manifest.parquet`, `eyegaze/manifest.report.json`.

In [ ]:
# --- Settings -------------------------------------------------------------
REPO_URL = "https://github.com/viettt0119/core_nckh.git"   # private repo: use https://<TOKEN>@github.com/...
BRANCH = "claude/exciting-johnson-bfrv9s"
ARCHIVE = "/content/drive/MyDrive/archive.zip"              # GazeCapture zip on Drive
DRIVE_OUT = "/content/drive/MyDrive/eyegaze"                # persistent outputs
LOCAL_DATA = "/content/gazecapture"                         # fast local copy of the frames
FRAME_STRIDE = 1                                            # 2 halves the work if time is short

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%%bash -s "$REPO_URL" "$BRANCH"
set -euo pipefail
apt-get -qq update && apt-get -qq install -y libgles2 libegl1 > /dev/null
pip -q install uv
if [ ! -d /content/core_nckh ]; then git clone -q --branch "$2" "$1" /content/core_nckh; else git -C /content/core_nckh pull -q; fi
cd /content/core_nckh && make setup > /dev/null && make models

In [ ]:
import os, subprocess
os.makedirs(DRIVE_OUT, exist_ok=True)
EYEGAZE = ["uv", "--directory", "/content/core_nckh/research", "run", "eyegaze"]
if not os.path.isdir(LOCAL_DATA) or not os.listdir(LOCAL_DATA):
    subprocess.run(EYEGAZE + ["data", "extract", "--zip", ARCHIVE, "--out", LOCAL_DATA], check=True)
print(os.listdir(LOCAL_DATA)[:5])

In [ ]:
workers = os.cpu_count() or 2
subprocess.run(EYEGAZE + [
    "data", "landmarks",
    "--data", LOCAL_DATA,
    "--out", f"{DRIVE_OUT}/landmarks",
    "--workers", str(workers),
    "--frame-stride", str(FRAME_STRIDE),
], check=True)

In [ ]:
import json
subprocess.run(EYEGAZE + [
    "data", "manifest",
    "--landmarks", f"{DRIVE_OUT}/landmarks",
    "--out", f"{DRIVE_OUT}/manifest.parquet",
], check=True)
print(json.dumps(json.load(open(f"{DRIVE_OUT}/manifest.report.json")), indent=2))